# Kubernetes Concepts for Scaling ML Deployments

You've built a model API in a container. Traffic spikes — your one container can't keep up. Kubernetes (K8s) orchestrates multiple replicas of your container, routes traffic between them, and automatically scales up or down based on load.

**Note:** `kubectl` runs against a live cluster and can't execute inside a Jupyter kernel. This notebook teaches K8s concepts through YAML examples and explains each command's effect. Run the commands in a terminal connected to a K8s cluster (Minikube, Kind, or cloud).

## Learning Objectives

By the end of this notebook you will be able to:
1. Explain what a Pod, Deployment, Service, and HPA are in plain terms
2. Read and write a Deployment YAML for an ML model container
3. Understand how a LoadBalancer Service routes traffic to Pods
4. Configure an HPA that scales 2-10 replicas based on CPU utilization
5. Explain zero-downtime rolling updates

## 1. The Problem Kubernetes Solves

Plain Docker: you run one container. If it crashes, it's gone. If traffic doubles, you manually start another. You have to load-balance them yourself.

Kubernetes:
- Keeps your desired number of replicas running at all times (if one crashes, K8s starts a replacement)
- Routes incoming traffic across all healthy replicas
- Watches CPU/memory usage and adds or removes replicas automatically
- Deploys new versions without any downtime

## 2. Key Concepts

**Pod** — the smallest K8s unit. Usually wraps one container. If a Pod crashes, K8s schedules a new one. Pods are ephemeral: don't store state in them.

**Deployment** — declares *how many* Pods to run and *which container image* to use. The Deployment controller continuously reconciles reality with the declaration.

**Service** — a stable network address (IP + DNS name) that routes traffic to a set of Pods. Pods come and go; the Service IP stays fixed.

**HorizontalPodAutoscaler (HPA)** — watches CPU (or custom metrics) and adjusts the Deployment's replica count automatically.

In [1]:
import os

YAML_DIR = "/tmp/k8s_manifests"
os.makedirs(YAML_DIR, exist_ok=True)
print(f"Writing K8s manifests to {YAML_DIR}")

Writing K8s manifests to /tmp/k8s_manifests


## 3. The Deployment YAML

A Deployment YAML is a declarative description of what you want to run. K8s continuously works to make the cluster match this declaration.

In [2]:
deployment_yaml = """\
apiVersion: apps/v1
kind: Deployment
metadata:
  name: iris-classifier
  labels:
    app: iris-classifier
spec:
  replicas: 3                          # Run 3 Pods simultaneously
  selector:
    matchLabels:
      app: iris-classifier             # This Deployment manages Pods with this label
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1                      # Allow 1 extra Pod during an update
      maxUnavailable: 0                # Never take a Pod down before a new one is ready
  template:
    metadata:
      labels:
        app: iris-classifier
    spec:
      containers:
        - name: iris-classifier
          image: myregistry/iris-classifier:v2   # Docker image to run
          ports:
            - containerPort: 8000
          resources:
            requests:
              cpu: "250m"              # 0.25 CPU cores requested (used for scheduling)
              memory: "256Mi"
            limits:
              cpu: "500m"              # Hard cap: container cannot use more than 0.5 cores
              memory: "512Mi"
          readinessProbe:              # K8s won't send traffic until this passes
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 10
            periodSeconds: 5
          livenessProbe:               # K8s restarts the container if this fails
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 30
            periodSeconds: 10
"""

with open(f"{YAML_DIR}/deployment.yaml", "w") as f:
    f.write(deployment_yaml)

print(deployment_yaml)

apiVersion: apps/v1
kind: Deployment
metadata:
  name: iris-classifier
  labels:
    app: iris-classifier
spec:
  replicas: 3                          # Run 3 Pods simultaneously
  selector:
    matchLabels:
      app: iris-classifier             # This Deployment manages Pods with this label
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1                      # Allow 1 extra Pod during an update
      maxUnavailable: 0                # Never take a Pod down before a new one is ready
  template:
    metadata:
      labels:
        app: iris-classifier
    spec:
      containers:
        - name: iris-classifier
          image: myregistry/iris-classifier:v2   # Docker image to run
          ports:
            - containerPort: 8000
          resources:
            requests:
              cpu: "250m"              # 0.25 CPU cores requested (used for scheduling)
              memory: "256Mi"
            limits:
              cpu: "500m"              # Hard cap: con

**Key fields to understand:**
- `replicas: 3` — K8s keeps exactly 3 Pods running. If one crashes, it starts a replacement.
- `resources.requests` — used by the scheduler to decide which node can fit this Pod.
- `resources.limits` — hard cap; container is killed if it exceeds this.
- `readinessProbe` — traffic only reaches a Pod once `/health` returns 200. This prevents requests going to a Pod that's still loading its model.
- `livenessProbe` — if `/health` fails repeatedly, K8s restarts the container.

## 4. The Service YAML

A Service gives your Pods a stable address. `LoadBalancer` type provisions a cloud load balancer (in AWS/GCP/Azure) that distributes traffic across all 3 Pods.

In [3]:
service_yaml = """\
apiVersion: v1
kind: Service
metadata:
  name: iris-classifier-svc
spec:
  type: LoadBalancer                   # Provisions a cloud load balancer with an external IP
  selector:
    app: iris-classifier               # Routes to any Pod with this label
  ports:
    - name: http
      port: 80                         # External port (clients call :80)
      targetPort: 8000                 # Internal container port (your FastAPI app)
      protocol: TCP
"""

with open(f"{YAML_DIR}/service.yaml", "w") as f:
    f.write(service_yaml)

print(service_yaml)
print("After applying this Service, clients call: http://<EXTERNAL-IP>/predict")
print("K8s load-balances across all 3 Pod replicas automatically.")

apiVersion: v1
kind: Service
metadata:
  name: iris-classifier-svc
spec:
  type: LoadBalancer                   # Provisions a cloud load balancer with an external IP
  selector:
    app: iris-classifier               # Routes to any Pod with this label
  ports:
    - name: http
      port: 80                         # External port (clients call :80)
      targetPort: 8000                 # Internal container port (your FastAPI app)
      protocol: TCP

After applying this Service, clients call: http://<EXTERNAL-IP>/predict
K8s load-balances across all 3 Pod replicas automatically.


## 5. The HPA — HorizontalPodAutoscaler

In [4]:
hpa_yaml = """\
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: iris-classifier-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: iris-classifier              # The Deployment this HPA controls
  minReplicas: 2                       # Never scale below 2 (minimum availability)
  maxReplicas: 10                      # Never scale above 10 (cost cap)
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: 70       # Scale up when average CPU across Pods > 70%
                                       # Scale down when average CPU drops below 70%
"""

with open(f"{YAML_DIR}/hpa.yaml", "w") as f:
    f.write(hpa_yaml)

print(hpa_yaml)
print("HPA behavior:")
print("  CPU > 70%  → add Pods (up to maxReplicas=10)")
print("  CPU < 70%  → remove Pods (down to minReplicas=2)")
print("  Scale-down has a 5-minute cooldown to avoid flapping")

apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: iris-classifier-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: iris-classifier              # The Deployment this HPA controls
  minReplicas: 2                       # Never scale below 2 (minimum availability)
  maxReplicas: 10                      # Never scale above 10 (cost cap)
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: 70       # Scale up when average CPU across Pods > 70%
                                       # Scale down when average CPU drops below 70%

HPA behavior:
  CPU > 70%  → add Pods (up to maxReplicas=10)
  CPU < 70%  → remove Pods (down to minReplicas=2)
  Scale-down has a 5-minute cooldown to avoid flapping


## 6. Key kubectl Commands

Run these in a terminal connected to your cluster (after `kubectl config use-context <cluster>`).

In [5]:
kubectl_commands = """
# --- Deploy everything ---
kubectl apply -f deployment.yaml
kubectl apply -f service.yaml
kubectl apply -f hpa.yaml

# --- Check Pod status ---
kubectl get pods
# NAME                              READY   STATUS    RESTARTS   AGE
# iris-classifier-7d9f6b8c4-4xqzp   1/1     Running   0          2m
# iris-classifier-7d9f6b8c4-k9v2d   1/1     Running   0          2m
# iris-classifier-7d9f6b8c4-rj3tf   1/1     Running   0          2m

# --- See the external IP for the Service ---
kubectl get service iris-classifier-svc
# NAME                    TYPE           CLUSTER-IP     EXTERNAL-IP     PORT(S)        AGE
# iris-classifier-svc     LoadBalancer   10.96.142.55   34.117.12.200   80:32100/TCP   3m

# --- Tail logs from one Pod ---
kubectl logs -f iris-classifier-7d9f6b8c4-4xqzp

# --- Watch the HPA scaling decisions ---
kubectl get hpa iris-classifier-hpa --watch
# NAME                   REFERENCE                      TARGETS   MINPODS   MAXPODS   REPLICAS
# iris-classifier-hpa    Deployment/iris-classifier     42%/70%   2         10        3

# --- Manual scale override ---
kubectl scale deployment iris-classifier --replicas=5

# --- Check rollout status after updating the image ---
kubectl rollout status deployment/iris-classifier
# Waiting for rollout to finish: 1 out of 3 new replicas have been updated...
# Waiting for rollout to finish: 2 out of 3 new replicas have been updated...
# deployment "iris-classifier" successfully rolled out

# --- Roll back if the new version is broken ---
kubectl rollout undo deployment/iris-classifier
"""

print(kubectl_commands)


# --- Deploy everything ---
kubectl apply -f deployment.yaml
kubectl apply -f service.yaml
kubectl apply -f hpa.yaml

# --- Check Pod status ---
kubectl get pods
# NAME                              READY   STATUS    RESTARTS   AGE
# iris-classifier-7d9f6b8c4-4xqzp   1/1     Running   0          2m
# iris-classifier-7d9f6b8c4-k9v2d   1/1     Running   0          2m
# iris-classifier-7d9f6b8c4-rj3tf   1/1     Running   0          2m

# --- See the external IP for the Service ---
kubectl get service iris-classifier-svc
# NAME                    TYPE           CLUSTER-IP     EXTERNAL-IP     PORT(S)        AGE
# iris-classifier-svc     LoadBalancer   10.96.142.55   34.117.12.200   80:32100/TCP   3m

# --- Tail logs from one Pod ---
kubectl logs -f iris-classifier-7d9f6b8c4-4xqzp

# --- Watch the HPA scaling decisions ---
kubectl get hpa iris-classifier-hpa --watch
# NAME                   REFERENCE                      TARGETS   MINPODS   MAXPODS   REPLICAS
# iris-classifier-hpa    Deploym

## 7. Rolling Update — Zero-Downtime Deployments

When you update the image tag in the Deployment YAML (e.g., `iris-classifier:v2` → `iris-classifier:v3`), K8s performs a rolling update:
1. Starts 1 new Pod with the new image (`maxSurge: 1` → temporarily 4 Pods exist)
2. Waits until the new Pod passes its readinessProbe
3. Terminates 1 old Pod (back to 3 total, but now 1 old + 2 new)
4. Repeats until all 3 are running the new version

Because `maxUnavailable: 0`, traffic is never sent to fewer than 3 ready Pods. Users never see downtime.

In [6]:
# Visualize the rolling update timeline
import time

print("Rolling update timeline (replicas=3, maxSurge=1, maxUnavailable=0):")
print()
print(f"{'Step':<6} {'Old Pods':^12} {'New Pods':^12} {'Total':^8} {'Traffic goes to'}")
print("-" * 62)

steps = [
    ("0",  3, 0, "old pods only"),
    ("1",  3, 1, "old pods only (new pod not ready yet)"),
    ("2",  2, 1, "2 old + 1 new (new pod passed readiness)"),
    ("3",  2, 2, "2 old + 2 new"),
    ("4",  1, 2, "1 old + 2 new"),
    ("5",  1, 3, "1 old + 3 new"),
    ("6",  0, 3, "new pods only"),
]

for step, old, new, note in steps:
    total = old + new
    print(f"{step:<6} {old:^12} {new:^12} {total:^8} {note}")

print()
print("At every step, at least 3 ready Pods serve traffic.")
print("Zero requests are dropped during the entire update.")

Rolling update timeline (replicas=3, maxSurge=1, maxUnavailable=0):

Step     Old Pods     New Pods    Total   Traffic goes to
--------------------------------------------------------------
0           3            0          3     old pods only
1           3            1          4     old pods only (new pod not ready yet)
2           2            1          3     2 old + 1 new (new pod passed readiness)
3           2            2          4     2 old + 2 new
4           1            2          3     1 old + 2 new
5           1            3          4     1 old + 3 new
6           0            3          3     new pods only

At every step, at least 3 ready Pods serve traffic.
Zero requests are dropped during the entire update.


## 8. Writing All Manifests to Disk

In [7]:
# Verify all three YAML files were written
import os
for fname in os.listdir(YAML_DIR):
    path = os.path.join(YAML_DIR, fname)
    size = os.path.getsize(path)
    print(f"{fname}: {size} bytes")

print(f"\nApply all with: kubectl apply -f {YAML_DIR}/")
print("K8s will create the Deployment, Service, and HPA in that order.")

deployment.yaml: 1511 bytes
service.yaml: 458 bytes
hpa.yaml: 675 bytes

Apply all with: kubectl apply -f /tmp/k8s_manifests/
K8s will create the Deployment, Service, and HPA in that order.


## 9. K8s vs Plain Docker

| Capability | Plain Docker | Kubernetes |
|---|---|---|
| Run 1 container | Yes | Yes |
| Auto-restart on crash | No (need docker restart policy) | Yes (always) |
| Load balance across replicas | No (need nginx/haproxy) | Yes (Service) |
| Auto-scale on CPU | No | Yes (HPA) |
| Zero-downtime update | No | Yes (rolling update) |
| Multi-node scheduling | No | Yes |
| Health-gate traffic | No | Yes (readinessProbe) |

## 10. Summary

In this notebook you:
- Learned the four core K8s objects: Pod, Deployment, Service, HPA
- Wrote a Deployment YAML with resource limits, readiness probes, and a rolling update strategy
- Wrote a LoadBalancer Service YAML that routes port 80 to container port 8000
- Wrote an HPA YAML that scales 2-10 replicas at 70% CPU target
- Traced through a rolling update showing zero Pods are unavailable at any step

**K8s in one sentence:** you declare what you want; K8s continuously makes the cluster match your declaration, handling failures and scaling automatically.

## Self-Check (answer before scrolling up)

1. **What is the difference between a Pod and a Deployment?** Why do you almost never create Pods directly?
2. **What does the HPA do when CPU drops below 70% for an extended period?** Is this instant? Why or why not?
3. **What is a rolling update and why is it better than stopping all replicas before starting new ones?** Which YAML field guarantees no Pod is taken down before a replacement is ready?